# Station Training Baseline — RJTT: Tokyo — Haneda Airport

**Status: active baseline.** This is the complete per-station workflow: the **Tokyo V20 Asia no-peak aligned** point pipeline, followed in this same notebook by **Tokyo 1C Market Ordinal Probability Model**, the pure cumulative-threshold ordinal probability model. Versioned source notebooks remain reference-only; new station work starts here.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "asia_station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/asia_station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CITY_ID = "tokyo"
CITY_LABEL = "Tokyo"
STATION_ID = "RJTT"
TIMEZONE = "Asia/Tokyo"
DATA_ROOT = PROJECT_ROOT / "data" / "calibration" / "asia_11am"
OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_training_baseline" / "Tokyo"
PROVIDERS = ("gfs", "gefs", "jma_msm")
TIMING_MODE = "asia_same_day_11am_live_safe"
FEATURE_VERSION = "v20_asia_no_peak"
TRAINING_PROFILE = "v20_aligned"
TARGET_SOURCE = "wunderground_only"
TARGET_MODE = "remaining_warmup"
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
FAST_MODE = False
EXPORT_MODEL_WEIGHTS = True
EXPORT_LIVE_MODEL_WEIGHTS = False
POINT_EVALUATION_TRAIN_YEARS = (2022, 2025)
LIVE_POINT_MODEL_VERSION = "station_high_regressor_live_tokyo_no_peak_stack_2026"
PROBABILITY_MODEL_VERSION = "station_bucket_baseline_tokyo_1c_market_ordinal"
PROBABILITY_TARGET = "celsius_market_1c"
PROBABILITY_OUTPUT_SUBDIR = "celsius_market_probability"
PROBABILITY_FEATURE_PROFILE = "asia_no_peak"
PROBABILITY_FEATURE_COUNT = 59
PROBABILITY_PROVIDERS = ('gfs', 'gefs', 'jma_msm')
PROBABILITY_DEVELOPMENT_YEARS = (2024, 2025)
PROBABILITY_FORWARD_VALIDATION_YEARS = (2025,)
PROBABILITY_HOLDOUT_YEAR = 2026
MODEL_VERSION = f"station_high_regressor_baseline_tokyo_no_peak_stack"
PROJECT_ROOT


In [2]:
import numpy as np
import pandas as pd

from src.calibration.asia_station_stacking import (
    ASIA_PROVIDERS,
    ASIA_TEST_YEAR,
    ASIA_TIMING_MODE,
    asia_expanding_folds,
    build_asia_station_wide_dataset,
    provider_readiness,
)
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_ASIA_NO_PEAK_FEATURE_VERSION,
    missing_model_dependencies,
    run_station_year_split_experiment,
)
from src.export_station_stacking_v2_models import export_station_model_weights


## City contract

- Existing Asia parquet data rooted at `data/calibration/asia_11am`
- Local 11 AM live-safe observation cutoff
- GFS, GEFS, and JMA MSM forecast inputs
- Wunderground-only daily settlement high target
- Fahrenheit-native model values with Celsius reporting


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
            "validation_weight": 1.0,
        }
        for fold in asia_expanding_folds()
    ]
)
fold_spec


,fold,train_start_year,train_end_year,validation_year,validation_weight
0,fold_2022_to_2023,2022,2022,2023,1.0
1,fold_2022_2023_to_2024,2022,2023,2024,1.0
2,fold_2022_2024_to_2025,2022,2024,2025,1.0


## Provider readiness


In [4]:
readiness = provider_readiness(DATA_ROOT, CITY_ID, providers=PROVIDERS)
readiness


,city_id,provider,row_count,ok_count,first_contract_date,last_contract_date,ready
0,tokyo,gfs,19318,19318,2022-07-03,2026-07-27,True
1,tokyo,gefs,1484,1484,2022-07-03,2026-07-25,True
2,tokyo,jma_msm,19318,19318,2022-07-03,2026-07-27,True


## Build the live-safe modeling frame


In [5]:
features = build_asia_station_wide_dataset(
    DATA_ROOT,
    CITY_ID,
    feature_version=FEATURE_VERSION,
    providers=PROVIDERS,
)
features[[
    "contract_date",
    "actual_high_f",
    "observed_high_temp_through_as_of_f",
    "gfs_high_f",
    "gefs_high_f",
    "jma_msm_high_f",
    "strict_quality_ok",
]].head()


,contract_date,actual_high_f,observed_high_temp_through_as_of_f,gfs_high_f,gefs_high_f,jma_msm_high_f,strict_quality_ok
0,2022-07-03,91.4,91.4,85.554000,91.255111,81.68,True
1,2022-07-04,86.0,82.4,80.115527,85.319289,79.70,True
2,2022-07-05,89.6,89.6,78.667182,84.022496,79.88,True
3,2022-07-06,84.2,82.4,86.573266,87.381659,76.64,True
4,2022-07-07,84.2,80.6,80.744440,87.374120,85.82,True


In [6]:
feature_coverage = (
    features[["actual_high_f", "observed_high_temp_through_as_of_f", *[f"{p}_high_f" for p in PROVIDERS]]]
    .notna()
    .mean()
    .rename("non_null_fraction")
    .to_frame()
)
feature_coverage


,non_null_fraction
actual_high_f,1.000000
observed_high_temp_through_as_of_f,0.998654
gfs_high_f,1.000000
gefs_high_f,0.998654
jma_msm_high_f,1.000000


## Train and score


In [7]:
missing_packages = missing_model_dependencies(("xgboost", "lightgbm", "catboost", "optuna"))
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    optuna_verbose=True,
    optuna_storage_path=OUTPUT_DIR / "RJTT_optuna_no_fullday_high.sqlite3",
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric="mae_f",
    feature_version=FEATURE_VERSION,
    training_profile=TRAINING_PROFILE,
    target_mode=TARGET_MODE,
    target_source=TARGET_SOURCE,
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=asia_expanding_folds(),
    year_split_validation_weights={2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2022, 2025),
    year_split_test_year=ASIA_TEST_YEAR,
    output_dir=OUTPUT_DIR,
    prebuilt_features=features,
)
config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_optuna_no_fullday_high.sqlite3')

In [8]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-08-05 23:18:09,057] A new study created in RDB with name: RJTT_v20_asia_no_peak_remaining_warmup_v20_aligned_base_xgboost_mae_f_wide
[I 2026-08-05 23:18:18,829] Trial 0 finished with value: 1.3123116833747808 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.3123116833747808.
[I 2026-08-05 23:18:56,733] Trial 1 finished with value: 1.4637135501889607 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 0 with value: 1.3123116833747808.
[I

,period,method,count,mae_f,rmse_f
0,validation_2023_2025,xgboost,1096,1.184123,1.580240
1,validation_2023_2025,lightgbm,1096,1.213334,1.663131
2,validation_2023_2025,catboost,1096,1.197301,1.595132
3,validation_2023_2025,provider_mean,1096,2.421523,2.979478
4,validation_2023_2025,provider_median,1096,2.526813,3.137402
5,validation_2023_2025,gfs_raw,1096,3.881351,4.677215
6,test_2026,xgboost,206,1.105639,1.492348
7,test_2026,lightgbm,206,1.082010,1.469916
8,test_2026,catboost,206,1.078707,1.438690
9,test_2026,ridge_stack,206,1.075406,1.462146


## Celsius reporting and export


In [9]:
celsius_predictions = result.test_predictions.copy()
for column in ("actual_high_f", "predicted_high_f"):
    if column in celsius_predictions:
        celsius_predictions[column.replace("_f", "_c")] = (
            pd.to_numeric(celsius_predictions[column], errors="coerce") - 32.0
        ) * 5.0 / 9.0
if "error_f" in celsius_predictions:
    celsius_predictions["error_c"] = (
        pd.to_numeric(celsius_predictions["error_f"], errors="coerce") * 5.0 / 9.0
    )
celsius_predictions.head()


,contract_date,fold,method,param_key,evaluation_scope,actual_high_f,predicted_high_f,error_f,absolute_error_f,actual_high_c,predicted_high_c,error_c
0,2026-01-01,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,48.2,48.074844,0.125156,0.125156,26.777778,26.708247,0.069531
1,2026-01-02,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,48.2,43.493077,4.706923,4.706923,26.777778,24.162821,2.614957
2,2026-01-03,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,48.2,47.773264,0.426736,0.426736,26.777778,26.540702,0.237075
3,2026-01-04,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,51.8,50.219355,1.580645,1.580645,28.777778,27.899642,0.878136
4,2026-01-05,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,57.2,54.638171,2.561829,2.561829,31.777778,30.354540,1.423238


In [ ]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        train_years=POINT_EVALUATION_TRAIN_YEARS,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        max_feature_missing_fraction=config.effective_max_feature_missing_fraction,
        source_pipeline="notebooks/station_training_baseline/stations/Tokyo",
    )
    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this notebook.")

if EXPORT_MODEL_WEIGHTS:
    import json as _point_export_json

    evaluation_point_manifest = _point_export_json.loads(
        exported_weights.manifest_path.read_text(encoding="utf-8")
    )
    assert evaluation_point_manifest["model_version"] == MODEL_VERSION
    assert evaluation_point_manifest["training"]["train_start_year"] == POINT_EVALUATION_TRAIN_YEARS[0]
    assert evaluation_point_manifest["training"]["train_end_year"] == POINT_EVALUATION_TRAIN_YEARS[1]
    assert evaluation_point_manifest["model_contract"]["max_feature_missing_fraction"] == config.effective_max_feature_missing_fraction


## Optional live-production point bundle

The evaluation bundle above is frozen before the exploratory holdout and is the
only point bundle used by probability training and holdout reporting. A live
production refit may use all completed actuals, including completed holdout-year
dates, but it has a distinct version and cannot claim holdout performance as
out-of-sample evidence. Keep this export disabled until the source is committed;
then create the immutable release record in a separate promotion review.


In [ ]:
live_exported_weights = None
if EXPORT_LIVE_MODEL_WEIGHTS:
    live_exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID if "CITY_ID" in globals() else None,
        artifact_dir=config.resolved_output_dir(),
        model_version=LIVE_POINT_MODEL_VERSION,
        train_years=None,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        max_feature_missing_fraction=config.effective_max_feature_missing_fraction,
        source_pipeline="notebooks/station_training_baseline/stations/Tokyo",
    )
    assert live_exported_weights.bundle_path != exported_weights.bundle_path
    assert LIVE_POINT_MODEL_VERSION != MODEL_VERSION
    print(
        "Live bundle exported as an unreleased candidate. "
        "Create a clean-checkout release record before promotion."
    )
else:
    print("Live-production export disabled; evaluation bundle remains frozen.")


In [11]:
result.output_paths


{'features': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_features.csv'),
 'year_split_tuning': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_year_split_tuning.csv'),
 'year_split_validation_predictions': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_year_split_validation_predictions.csv'),
 'year_split_test_predictions': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_year_split_test_predictions.csv'),
 'year_split_metrics': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_year_split_metrics.csv'),
 'year_split_selected_hyperparameters': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/RJTT_year_split_selected_hyperparameters.csv'),
 'year_split_feature_importance': WindowsPath('D:/dev/weather-research/data/calibration/station_trai

## Tokyo 1C Market Ordinal Probability Model — market-aligned correction

This stage replaces Tokyo's historical integer-Fahrenheit/2°F probability
target with the actual Tokyo Polymarket whole-1°C market contract. The point
model remains Fahrenheit-native.

- `point_prediction_c = (point_prediction_f - 32) * 5 / 9`;
- `point_bucket_c = floor(point_prediction_c + 0.5)`;
- `actual_bucket_c = floor(actual_high_c + 0.5)`;
- `offset_c = actual_bucket_c - point_bucket_c`;
- ordered classes: `<=-3, -2, -1, 0, +1, +2, >=+3` °C, chosen from pre-2026
  support with open tails;
- exact market probabilities are `point_bucket_c + exact_offset_c` and sum to
  one on every row;
- all confidence thresholds and the tail-ambiguity rule are selected on the
  [2025] forward-validation rows only;
- 2026 remains exploratory and cannot select the model or policy.

The source frame has no settlement-equivalent `actual_high_c` or
`settlement_high_c` field. Its target is Wunderground `actual_high_f`, while
`iem_daily_high_c` is diagnostic and a different source. Therefore the Celsius
target uses the exact Fahrenheit-to-Celsius conversion fallback. This matches
Tokyo Polymarket's integer Celsius settlement buckets without approximating the
old 2°F distribution.


In [12]:
import json

from src.calibration.celsius_market_probability import (
    OFFSET_LABELS_C,
    TARGET_CONTRACT,
    build_celsius_probability_frame,
    celsius_calibration_table,
    celsius_probability_metrics,
    evaluate_celsius_probability_holdout,
    export_celsius_probability_bundle,
    fit_celsius_probability_system,
    sha256_file as celsius_sha256_file,
)
from src.calibration.bucket_probability import probability_feature_names
from src.calibration.v19_bucket import crossfit_ridge_predictions


### Celsius feature and target contract


In [13]:
celsius_feature_names = probability_feature_names(
    include_peak_features=False,
    feature_profile=PROBABILITY_FEATURE_PROFILE,
)
celsius_feature_contract = pd.DataFrame(
    {"position": range(1, len(celsius_feature_names) + 1), "feature": celsius_feature_names}
)
celsius_target_contract = {
    "market": "Tokyo Polymarket whole 1C integer buckets",
    "rounding": "round_half_up(value) = floor(value + 0.5)",
    "target": TARGET_CONTRACT,
    "actual_celsius_source_used": "actual_high_f_converted_to_c",
    "excluded_diagnostic_source": "iem_daily_high_c",
    "ordered_offset_classes_c": list(OFFSET_LABELS_C),
    "tail_contract": "training-supported exact offsets within <=-3 and >=+3",
    "feature_profile": PROBABILITY_FEATURE_PROFILE,
    "feature_count": len(celsius_feature_names),
    "providers": list(PROBABILITY_PROVIDERS),
}
assert len(celsius_feature_names) == PROBABILITY_FEATURE_COUNT
celsius_target_contract


{'market': 'Tokyo Polymarket whole 1C integer buckets',
 'rounding': 'round_half_up(value) = floor(value + 0.5)',
 'target': 'point_bucket_c=round_half_up((point_prediction_f-32)*5/9); actual_bucket_c=round_half_up(actual_high_c); offset_c=actual_bucket_c-point_bucket_c',
 'actual_celsius_source_used': 'actual_high_f_converted_to_c',
 'excluded_diagnostic_source': 'iem_daily_high_c',
 'ordered_offset_classes_c': ['<=-3', '-2', '-1', '0', '+1', '+2', '>=+3'],
 'tail_contract': 'training-supported exact offsets within <=-3 and >=+3',
 'feature_profile': 'asia_no_peak',
 'feature_count': 59,
 'providers': ['gfs', 'gefs', 'jma_msm']}

### Fit with chronological [2025] outer validation


In [14]:
point_forward_predictions = crossfit_ridge_predictions(
    result.validation_predictions,
    providers=PROBABILITY_PROVIDERS,
)
assert not point_forward_predictions.empty
assert (
    point_forward_predictions["train_through_year"]
    < point_forward_predictions["validation_year"]
).all()

celsius_training_frame = build_celsius_probability_frame(
    result.features,
    point_forward_predictions,
    result.validation_predictions,
    include_peak_features=False,
    feature_profile=PROBABILITY_FEATURE_PROFILE,
)
assert celsius_training_frame["actual_high_c_source"].eq(
    "actual_high_f_converted_to_c"
).all()
celsius_bundle, celsius_forward_predictions, celsius_tuning = (
    fit_celsius_probability_system(
        celsius_training_frame,
        station_id=STATION_ID,
        point_model_version=MODEL_VERSION,
        point_bundle_sha256=celsius_sha256_file(exported_weights.bundle_path),
        feature_profile=PROBABILITY_FEATURE_PROFILE,
        model_version=PROBABILITY_MODEL_VERSION,
        development_years=PROBABILITY_DEVELOPMENT_YEARS,
        forward_validation_years=PROBABILITY_FORWARD_VALIDATION_YEARS,
    )
)
assert celsius_bundle["selected_family"] == "celsius_offset_ordinal_logistic"
assert celsius_bundle["training_cutoff"] < f"{PROBABILITY_HOLDOUT_YEAR}-01-01"
celsius_probability_metrics(celsius_forward_predictions)


,offset_log_loss,offset_brier,ranked_probability_score,offset_accuracy,offset_top_two_accuracy,offset_calibration_error,count,market_bucket_accuracy,point_bucket_accuracy,market_bucket_log_loss,market_bucket_brier,decision_coverage,decision_count,decision_accuracy
0,1.349116,0.651838,0.071779,0.484932,0.778082,0.043289,365,0.484932,0.468493,1.350309,0.650979,0.569863,208,0.5625


### Verify chronology and exact Celsius market probabilities


In [15]:
celsius_forward_dates = pd.to_datetime(celsius_forward_predictions["contract_date"])
assert set(celsius_forward_predictions["validation_year"]) == set(
    PROBABILITY_FORWARD_VALIDATION_YEARS
)
assert (
    pd.to_datetime(celsius_forward_predictions["model_training_cutoff"])
    < celsius_forward_dates
).all()
assert (
    pd.to_datetime(celsius_forward_predictions["calibration_training_cutoff"])
    < pd.to_datetime(celsius_forward_predictions["calibration_validation_start"])
).all()
assert (
    pd.to_datetime(celsius_forward_predictions["calibration_validation_cutoff"])
    < celsius_forward_dates
).all()
for column in ("celsius_offset_probabilities", "market_bucket_probabilities_c"):
    assert celsius_forward_predictions[column].map(
        lambda probabilities: np.isclose(sum(probabilities.values()), 1.0, atol=1e-10)
    ).all()
assert celsius_forward_predictions.apply(
    lambda row: int(row["recommended_bucket_c"])
    == int(max(row["market_bucket_probabilities_c"], key=row["market_bucket_probabilities_c"].get)),
    axis=1,
).all()
assert celsius_bundle["policy_selection_data"] == "pre-2026 forward validation only"
celsius_bundle["decision_thresholds"]


{'minimum_top_probability': 0.39999999999999997,
 'minimum_top_two_margin': 0.125,
 'minimum_switch_advantage': 0.15000000000000002,
 'tail_ambiguity_rule_enabled': True,
 'target_coverage': 0.6}

### Evaluate the frozen model on exploratory 2026


In [16]:
holdout_point_predictions = result.test_predictions.loc[
    result.test_predictions["method"].eq("ridge_stack"),
    ["contract_date", "actual_high_f", "predicted_high_f"],
].copy()
assert not holdout_point_predictions.empty

celsius_holdout_predictions, celsius_holdout_metrics, celsius_holdout_calibration = (
    evaluate_celsius_probability_holdout(
        result.features,
        holdout_point_predictions,
        result.test_predictions,
        celsius_bundle,
        holdout_year=PROBABILITY_HOLDOUT_YEAR,
    )
)
assert not celsius_holdout_predictions.empty
assert pd.to_datetime(celsius_holdout_predictions["contract_date"]).dt.year.eq(
    PROBABILITY_HOLDOUT_YEAR
).all()
for column in ("celsius_offset_probabilities", "market_bucket_probabilities_c"):
    assert celsius_holdout_predictions[column].map(
        lambda probabilities: np.isclose(sum(probabilities.values()), 1.0, atol=1e-10)
    ).all()
celsius_bundle["holdout_metrics"] = celsius_holdout_metrics.iloc[0].to_dict()
celsius_bundle["holdout_status"] = "exploratory_previously_inspected_shadow_only"
celsius_holdout_metrics


,offset_log_loss,offset_brier,ranked_probability_score,offset_accuracy,offset_top_two_accuracy,offset_calibration_error,count,market_bucket_accuracy,point_bucket_accuracy,market_bucket_log_loss,market_bucket_brier,decision_coverage,decision_count,decision_accuracy
0,1.260058,0.645879,0.07227,0.519417,0.718447,0.121319,206,0.519417,0.553398,1.2619,0.645745,0.475728,98,0.622449


### Export the isolated Celsius research artifacts


In [17]:
celsius_output_dir = config.resolved_output_dir() / PROBABILITY_OUTPUT_SUBDIR
celsius_output_dir.mkdir(parents=True, exist_ok=True)

def _serialize_probability_columns(frame):
    output = frame.copy()
    for column in ("celsius_offset_probabilities", "market_bucket_probabilities_c"):
        output[column] = output[column].map(lambda value: json.dumps(value, sort_keys=True))
    return output

celsius_artifact_paths = []
forward_predictions_path = celsius_output_dir / f"{STATION_ID}_forward_validation_predictions.csv"
forward_metrics_path = celsius_output_dir / f"{STATION_ID}_forward_validation_metrics.csv"
holdout_predictions_path = celsius_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_predictions.csv"
holdout_metrics_path = celsius_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_metrics.csv"
forward_calibration_path = celsius_output_dir / f"{STATION_ID}_forward_validation_calibration.csv"
holdout_calibration_path = celsius_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_calibration.csv"
tuning_path = celsius_output_dir / f"{STATION_ID}_pre_2026_tuning.csv"
feature_contract_path = celsius_output_dir / f"{STATION_ID}_celsius_feature_contract.csv"
target_contract_path = celsius_output_dir / f"{STATION_ID}_celsius_target_contract.json"

_serialize_probability_columns(celsius_forward_predictions).to_csv(forward_predictions_path, index=False)
celsius_probability_metrics(celsius_forward_predictions).to_csv(forward_metrics_path, index=False)
_serialize_probability_columns(celsius_holdout_predictions).to_csv(holdout_predictions_path, index=False)
celsius_holdout_metrics.to_csv(holdout_metrics_path, index=False)
celsius_calibration_table(celsius_forward_predictions).to_csv(forward_calibration_path, index=False)
celsius_holdout_calibration.to_csv(holdout_calibration_path, index=False)
celsius_tuning.to_csv(tuning_path, index=False)
celsius_feature_contract.to_csv(feature_contract_path, index=False)
target_contract_path.write_text(json.dumps(celsius_target_contract, indent=2, sort_keys=True) + "\n", encoding="utf-8")
celsius_artifact_paths.extend([
    forward_predictions_path, forward_metrics_path, holdout_predictions_path,
    holdout_metrics_path, forward_calibration_path, holdout_calibration_path,
    tuning_path, feature_contract_path, target_contract_path,
])

celsius_bundle_path, celsius_manifest_path = export_celsius_probability_bundle(
    celsius_bundle,
    celsius_output_dir / "model_weights",
    source_identity={
        "pipeline": "station_training_baseline",
        "notebook": "notebooks/station_training_baseline/stations/Tokyo/train_Tokyo.ipynb",
        "point_workflow": "Tokyo V20 Asia no-peak aligned",
        "probability_setup": "Tokyo 1C Market Ordinal Probability Model",
    },
    artifact_paths=celsius_artifact_paths,
)
celsius_manifest = json.loads(celsius_manifest_path.read_text(encoding="utf-8"))
assert celsius_manifest["point_bundle_sha256"] == celsius_sha256_file(exported_weights.bundle_path)
assert celsius_manifest["artifact_integrity"]["bundle_sha256"] == celsius_sha256_file(celsius_bundle_path)
for path in celsius_artifact_paths:
    assert celsius_manifest["artifact_integrity"]["artifact_sha256"][path.name] == celsius_sha256_file(path)
{
    "point_bundle": exported_weights.bundle_path,
    "point_manifest": exported_weights.manifest_path,
    "celsius_probability_bundle": celsius_bundle_path,
    "celsius_probability_manifest": celsius_manifest_path,
    "output_dir": celsius_output_dir,
}


{'point_bundle': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/model_weights/RJTT_station_high_regressor_baseline_tokyo_no_peak_stack.joblib'),
 'point_manifest': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/model_weights/RJTT_station_high_regressor_baseline_tokyo_no_peak_stack.json'),
 'celsius_probability_bundle': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/celsius_market_probability/model_weights/RJTT_station_bucket_baseline_tokyo_1c_market_ordinal.joblib'),
 'celsius_probability_manifest': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/celsius_market_probability/model_weights/RJTT_station_bucket_baseline_tokyo_1c_market_ordinal.json'),
 'output_dir': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/Tokyo/celsius_market_probability')}